# Resample Train Set — 提高 Multinodule 比例

產生 `train_v5_resample.txt`，對 multinodule 圖片做 oversample，讓 multinodule 在訓練集中佔更高比例。

In [1]:
from pathlib import Path
import random

# ── 路徑設定 ──────────────────────────────────────────────────────────────────
OD_DIR      = Path("/Users/tony.tu/Desktop/戴承智慧/thyroid/yolo_data/all_data_nodule_normal_cut_seg")
LABEL_DIR   = OD_DIR / "labels"
TRAIN_TXT   = OD_DIR / "train_v5.txt"
OUTPUT_TXT  = OD_DIR / "train_v5_resample.txt"

# ── 參數設定 ──────────────────────────────────────────────────────────────────
MULTINODULE_REPEAT = 3   # multinodule 圖片重複幾次（可調整）
RANDOM_SEED        = 42

random.seed(RANDOM_SEED)

# ── 讀取原始 train list ────────────────────────────────────────────────────────
train_paths = [line.strip() for line in TRAIN_TXT.read_text().splitlines() if line.strip()]
print(f"原始 train 數量: {len(train_paths)}")

原始 train 數量: 16167


In [2]:
def get_nodule_count(img_path: str) -> int:
    """根據圖片路徑找到對應的 label 檔，回傳 bbox 數量（行數）。"""
    stem = Path(img_path).stem          # e.g. "train_1922"
    label_file = LABEL_DIR / f"{stem}.txt"
    if not label_file.exists():
        return 0
    lines = [l for l in label_file.read_text().splitlines() if l.strip()]
    return len(lines)

# ── 分類：normal / single / multinodule ──────────────────────────────────────
normal_paths      = []
single_paths      = []
multinodule_paths = []

for p in train_paths:
    n = get_nodule_count(p)
    if n == 0:
        normal_paths.append(p)
    elif n == 1:
        single_paths.append(p)
    else:
        multinodule_paths.append(p)

print(f"  normal      : {len(normal_paths)}")
print(f"  single      : {len(single_paths)}")
print(f"  multinodule : {len(multinodule_paths)}")
print(f"  合計         : {len(normal_paths)+len(single_paths)+len(multinodule_paths)}")

  normal      : 3889
  single      : 10085
  multinodule : 2193
  合計         : 16167


In [3]:
# ── 建立 resample list ────────────────────────────────────────────────────────
# 策略：multinodule 重複 MULTINODULE_REPEAT 次，其餘保持原樣，最後 shuffle
resampled = (
    normal_paths
    + single_paths
    + multinodule_paths * MULTINODULE_REPEAT
)

random.shuffle(resampled)

print(f"\nResample 後總數: {len(resampled)}")
print(f"  multinodule 佔比: {len(multinodule_paths)*MULTINODULE_REPEAT / len(resampled)*100:.1f}%")

# ── 寫出檔案 ──────────────────────────────────────────────────────────────────
OUTPUT_TXT.write_text("\n".join(resampled) + "\n")
print(f"\n已寫出: {OUTPUT_TXT}")


Resample 後總數: 20553
  multinodule 佔比: 32.0%

已寫出: /Users/tony.tu/Desktop/戴承智慧/thyroid/yolo_data/all_data_nodule_normal_cut_seg/train_v5_resample.txt


In [4]:
# ── 分佈驗證 ─────────────────────────────────────────────────────────────────
from collections import Counter

counts = Counter(get_nodule_count(p) for p in resampled)
print("Resample 後各類別數量 (nodule count → 圖片數):")
for k in sorted(counts):
    label = "normal" if k == 0 else ("single" if k == 1 else f"multi({k})")
    print(f"  {label:12s}: {counts[k]}")

Resample 後各類別數量 (nodule count → 圖片數):
  normal      : 3889
  single      : 10085
  multi(2)    : 4680
  multi(3)    : 1353
  multi(4)    : 390
  multi(5)    : 108
  multi(6)    : 27
  multi(7)    : 21
